In [3]:
# %% [markdown]
# # Exploratory Data Analysis for Cryptocurrency Data (BTC, ETH, LTC, USDT, XRP)
# # To be included/referenced in Thesis Methodology - Data Section

# %% [markdown]
# ## 1. Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import yfinance as yf
from statsmodels.tsa.stattools import adfuller
from scipy import stats # For skewness, kurtosis, probplot
import statsmodels.api as sm # For QQ plots

# Plotting Style Preferences (Optional)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# %% [markdown]
# ## 2. Configuration

# %%
# Define Cryptocurrencies and Time Period
tickers = ["BTC-USD"]
start_date = "2020-01-01"
end_date = "2025-01-01" # Matches model training data

# %% [markdown]
# ## 3. Data Loading Function

# %%
def load_crypto_data(ticker, start, end):
    """Downloads, cleans, and prepares data for a single crypto."""
    try:
        df = yf.download(tickers=[ticker], start=start, end=end)
        if df.empty:
            print(f"Warning: No data downloaded for {ticker}.")
            return None
        df.columns = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
        df = df[['Close']] # Select only Close price
        df = df.asfreq('D') # Ensure daily frequency
        df.ffill(inplace=True) # Forward fill missing days
        df.dropna(inplace=True) # Drop any remaining NaNs (e.g., at start)
        return df
    except Exception as e:
        print(f"Error downloading or processing {ticker}: {e}")
        return None

# %% [markdown]
# ## 4. Load and Combine Data

# %%
all_data = {}
for ticker in tickers:
    print(f"Loading data for {ticker}...")
    data = load_crypto_data(ticker, start_date, end_date)
    if data is not None:
        all_data[ticker] = data

# Combine into a single DataFrame
if not all_data:
    raise ValueError("No data loaded for any ticker. Check ticker symbols or date range.")

# Use the index of the first successfully loaded crypto as the base
base_ticker = list(all_data.keys())[0]
combined_df = pd.DataFrame(index=all_data[base_ticker].index)

for ticker, df in all_data.items():
    # Rename column before merging to avoid conflicts
    df_renamed = df.rename(columns={'Close': f'{ticker}_Close'})
    # Merge using outer join to keep all dates, then ffill again
    combined_df = pd.merge(combined_df, df_renamed, left_index=True, right_index=True, how='outer')

# Forward fill potential NaNs from merging different start dates/missing points
combined_df.ffill(inplace=True)
# Drop any rows where the base ticker might still be NaN (if it started later)
combined_df.dropna(subset=[f'{base_ticker}_Close'], inplace=True)

print("\n--- Combined Data Overview ---")
print(f"Shape: {combined_df.shape}")
print(f"Date Range: {combined_df.index.min()} to {combined_df.index.max()}")
print("Head:")
print(combined_df.head())
print("\nTail:")
print(combined_df.tail())
print("\nMissing Values Check:")
print(combined_df.isnull().sum())

# %% [markdown]
# ## 5. Price Visualization

# %%
print("\n--- Plotting Closing Prices ---")
combined_df.plot(figsize=(15, 8), linewidth=1.5)
plt.title('Daily Closing Prices (USD)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.yscale('log') # Use log scale due to large price differences
plt.legend(title='Cryptocurrency')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
# plt.savefig('closing_prices_log_scale.png') # Optional: Save plot
plt.show()

# Consider plotting without log scale for specific comparisons if needed
# combined_df.plot(figsize=(15, 8), secondary_y=['USDT-USD_Close']) # Example with secondary axis
# plt.title('Daily Closing Prices (USD) - Linear Scale')
# ... (add labels, legend etc.)
# plt.show()


# %% [markdown]
# ## 6. Return Calculation and Visualization

# %%
print("\n--- Calculating and Plotting Daily Log Returns ---")
# Calculate daily log returns: ln(P_t / P_{t-1})
returns_df = np.log(combined_df / combined_df.shift(1))
returns_df.dropna(inplace=True) # Remove first row with NaN

returns_df.plot(figsize=(15, 8), linewidth=1)
plt.title('Daily Log Returns')
plt.xlabel('Date')
plt.ylabel('Log Return')
plt.legend(title='Cryptocurrency')
plt.grid(True, linestyle='--', linewidth=0.5)
plt.axhline(0, color='black', linestyle='--', linewidth=0.7) # Add line at zero return
plt.tight_layout()
# plt.savefig('log_returns.png') # Optional: Save plot
plt.show()

# %% [markdown]
# ## 7. Descriptive Statistics

# %%
print("\n--- Descriptive Statistics for Closing Prices ---")
# Use .describe() and transpose for better readability
price_stats = combined_df.describe().T
print(price_stats.to_string()) # Use to_string to print full table

print("\n--- Descriptive Statistics for Daily Log Returns ---")
return_stats = returns_df.describe().T
print(return_stats.to_string())

print("\n--- Skewness and Kurtosis for Daily Log Returns ---")
for col in returns_df.columns:
    skew = stats.skew(returns_df[col])
    kurt = stats.kurtosis(returns_df[col]) # Fisher kurtosis (normal = 0)
    print(f"{col}: Skewness = {skew:.4f}, Kurtosis = {kurt:.4f}")

# %% [markdown]
# ## 8. Distribution Analysis (Log Returns)

# %%
print("\n--- Analyzing Distribution of Daily Log Returns ---")

num_plots = len(returns_df.columns)
num_cols = 2 # Adjust layout if needed
num_rows = int(np.ceil(num_plots / num_cols))

fig, axes = plt.subplots(num_rows, num_cols, figsize=(7 * num_cols, 5 * num_rows))
axes = axes.flatten() # Flatten axes array for easy iteration

for i, col in enumerate(returns_df.columns):
    if i < len(axes): # Check if we have enough axes
        ax = axes[i]
        # Histogram with KDE
        sns.histplot(returns_df[col], kde=True, ax=ax, stat='density', bins=50)
        # Overlay standard normal distribution for comparison
        xmin, xmax = ax.get_xlim()
        x_norm = np.linspace(xmin, xmax, 100)
        p_norm = stats.norm.pdf(x_norm, returns_df[col].mean(), returns_df[col].std()) # Fit normal to sample mean/std
        ax.plot(x_norm, p_norm, 'k', linewidth=1, linestyle='--', label='Normal Fit')
        ax.set_title(f'Distribution of {col} Log Returns')
        ax.set_xlabel('Log Return')
        ax.legend()

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
# plt.savefig('return_distributions.png') # Optional: Save plot
plt.show()

# Create QQ Plots
print("\n--- QQ Plots for Daily Log Returns (vs Normal Distribution) ---")
fig_qq, axes_qq = plt.subplots(num_rows, num_cols, figsize=(6 * num_cols, 5 * num_rows))
axes_qq = axes_qq.flatten()

for i, col in enumerate(returns_df.columns):
     if i < len(axes_qq):
          ax_qq = axes_qq[i]
          sm.qqplot(returns_df[col], line='s', ax=ax_qq) # 's' standardized line
          ax_qq.set_title(f'QQ Plot of {col} Log Returns')

# Hide any unused subplots
for j in range(i + 1, len(axes_qq)):
     fig_qq.delaxes(axes_qq[j])

plt.tight_layout()
# plt.savefig('return_qq_plots.png') # Optional: Save plot
plt.show()

# %% [markdown]
# ## 9. Stationarity Testing (ADF Test on Log Returns)

# %%
# Define the ADF test function
def adf_test(series, title=''):
    """Performs ADF test and prints results."""
    print(f'\nAugmented Dickey-Fuller Test: {title}')
    result = adfuller(series.dropna(), autolag='AIC')
    labels = ['ADF Test Statistic', 'p-value', '# Lags Used', '# Observations']
    out = pd.Series(result[0:4], index=labels)
    for key, val in result[4].items():
        out[f'Critical Value ({key})'] = val
    print(out.to_string())
    if result[1] <= 0.05:
        print("=> Conclusion: Likely Stationary (reject null hypothesis)")
    else:
        print("=> Conclusion: Likely Non-Stationary (fail to reject null hypothesis)")

print("\n--- Stationarity Test (ADF) on Daily Log Returns ---")
for col in returns_df.columns:
    adf_test(returns_df[col], title=col)

# %% [markdown]
# ## 10. Correlation Analysis (Log Returns)

# %%
print("\n--- Correlation Matrix of Daily Log Returns ---")
correlation_matrix = returns_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Daily Log Returns')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
# plt.savefig('return_correlation_heatmap.png') # Optional: Save plot
plt.show()

print("\nCorrelation Matrix Values:")
print(correlation_matrix)

# %% [markdown]
# --- End of Exploratory Data Analysis Script ---

[*********************100%***********************]  1 of 1 completed

Loading data for BTC-USD...
Error downloading or processing BTC-USD: Length mismatch: Expected axis has 5 elements, new values have 6 elements


ValueError: No data loaded for any ticker. Check ticker symbols or date range.